# 函数

> **开始前请注意：在线输入限制**
> 当前在线环境中的 `scanf`、`getchar`、`fgets(..., stdin)` 无法交互读取键盘输入。在线实验请修改变量的初值，或使用 `sscanf` 从字符串读取；键盘输入练习请在本地 GCC / Clang 中运行完整 C 程序。

**本章目标**：区分声明、定义和调用；传入数组及长度；设计明确的错误结果；验证递归边界。

**学习方法**：先预测输出，再运行验证；每次只修改一个条件，最后用自己的话解释变化。修改函数或类型定义后，请重启内核并从头运行。

在 C 语言中，**函数** 是完成特定任务的一段代码，其他代码可以通过函数名多次调用函数。  

它能提高代码的可重用性和可读性。

C 程序从 `main` 函数开始执行。

## 1. 函数定义

```c
返回值类型 函数名(参数列表) {
    // 函数体
    return 返回值;
}
```

## 2. 函数调用

In [ ]:
// 示例：求两个数的和
#include <stdio.h>

// 定义add函数
int add(int a, int b) {
    return a + b;
}

{
    //调用add函数
    int result = add(3, 5);
    printf("结果: %d\n", result);
}

## 3. 函数参数

在 C 中，函数参数都是“值传递”，函数获得的是实参的副本（copy），该副本被称为形参，对形参的修改不会影响实参（原变量）。

In [ ]:
void change(int x) { //此处x称为函数的形参
    x = 10;
    printf("x = %d\n", x);  // x = 10
}

{
    int a = 5;
    change(a); //此处，a称为实参
    printf("a = %d\n", a);  // a = 5 a没有改变
}

使用指针修改实参

In [ ]:
void change2(int *x) {
    *x = 10;
}

{
    int a = 5;
    change2(&a);   //scanf函数传参时与此相同，需要传入变量的地址
    printf("%d\n", a);  // 修改为 10
}

## 4. 函数返回值

函数可以返回一个值，也可以返回 void 表示无返回值。

In [ ]:
int square(int x) {
    return x * x;
}

void hello(int x) {
    printf("Hello, C!\n");
    printf("x = %d\n",x);
}

{
    int y = square(10);
    hello(y);
}

## 5. 变量作用域与存储期

- 块内局部变量的名字只在对应作用域中可见，普通自动局部对象在离开块后不再存在。
- 文件作用域名字通常从声明处开始可见；跨文件访问还涉及声明与链接，并非“所有函数都能随意访问”。
- 作用域回答“名字在哪里可见”，存储期回答“对象何时存在”。初学时优先通过参数和返回值传递数据。

In [ ]:
int global = 100; // 演示文件作用域；函数只读取，不修改它
void show_scope(void) {
    int local = 10;
    printf("local=%d, global=%d\n", local, global);
}
show_scope();

## 6. 递归函数

递归需要**终止条件**和**向终止条件推进的步骤**。先明确允许的输入，再讨论算法。

下面用 `long long` 返回阶乘，仅接受 0..20；20! 在 C17 保证的 long long 范围内，21! 不在本例允许范围。-1 明确表示无效输入，不能当成正常结果。斐波那契示例限制 1..20，避免指数级重复计算拖慢浏览器。

In [ ]:
long long factorial(int n) {
    if (n < 0 || n > 20) { return -1; }
    if (n == 0) { return 1; }
    return n * factorial(n - 1);
}

int fib(int n) {
    if (n < 1 || n > 20) { return -1; }
    if (n <= 2) { return 1; }
    return fib(n - 1) + fib(n - 2);
}

{
    printf("0!=%lld 5!=%lld\n", factorial(0), factorial(5));
    printf("20!=%lld\n", factorial(20));
    printf("无效输入: %lld %lld\n", factorial(-1), factorial(21));
    printf("fib(10)=%d\n", fib(10));
}

## 实验 1：先声明，后调用，再定义

声明告诉编译器参数和返回值类型，定义提供函数体。在同一个单元中先声明、再定义，然后调用；本地程序可以把定义放在 main 后面，只要调用处已能看到声明。

**预测**调用结果；修改声明而不修改定义，会发生什么？先在纸上判断，不要把不匹配版本混入后续运行。

In [ ]:
int larger(int left, int right);
int larger(int left, int right) { return left > right ? left : right; }
printf("较大值: %d\n", larger(85, 90));

## 实验 2：数组参数必须带长度

形参 `const int scores[]` 会调整为指针参数，函数不会自动获得调用者的数组长度。不要在函数内用 `sizeof(scores) / sizeof(scores[0])` 求人数。

`const` 表示此函数不通过该指针修改元素。调用者仍要保证实际数组至少有 count 个元素；函数无法从地址推断容量。

下面是贯穿案例的统计函数：最多 100 人，成绩必须在 0..100。`valid == 0` 明确表示输入无效，调用者必须检查。

In [ ]:
struct GradeSummary {
    int valid;
    double average;
    size_t passed;
};

// 不修改输入；count 必须对应调用者提供的实际元素数。
struct GradeSummary summarize(const int scores[], size_t count) {
    struct GradeSummary result = {0, 0.0, 0};
    if (scores == NULL || count == 0 || count > 100) { return result; }
    int total = 0;
    size_t passed = 0;
    for (size_t i = 0; i < count; ++i) {
        if (scores[i] < 0 || scores[i] > 100) { return result; }
        total += scores[i];
        if (scores[i] >= 60) { ++passed; }
    }
    result.valid = 1;
    result.average = (double)total / count;
    result.passed = passed;
    return result;
}

## 贯穿案例 6：分离统计与显示

**预测**平均分 77.67、及格人数 2。统计函数只返回结果，显示由调用代码完成。将第三人成绩改成 -1，应该进入错误分支，不能显示部分统计结果。

In [ ]:
{
    const int scores[] = {85, 90, 58};
    size_t count = sizeof scores / sizeof scores[0];
    struct GradeSummary result = summarize(scores, count);
    if (!result.valid) { printf("无法统计：人数或成绩无效\n"); }
    else { printf("平均分: %.2f, 及格: %zu/%zu\n", result.average, result.passed, count); }
}

## 边界实验：空数据与非法成绩

以下是有意传入的无效输入，函数应明确拒绝，不访问空指针，也不执行除以零。预测两个 valid 字段的值后运行。

In [ ]:
{
    const int invalid[] = {85, -1};
    struct GradeSummary empty = summarize(NULL, 0);
    struct GradeSummary bad = summarize(invalid, 2);
    printf("空数据有效=%d 非法成绩有效=%d\n", empty.valid, bad.valid);
}

## 实验 3：展开递归调用

先写下 `factorial(3)` 的调用链：3 → 2 → 1 → 0，然后从 0! = 1 向上返回。请用循环实现相同的 0..20 阶乘接口，比较输入 0、5、20、-1、21。

递归不一定更快；朴素 fib 会重复计算相同子问题。对连续生成斐波那契数列，可保留前两个结果并迭代。

## 分层练习

### 1. 读程序
函数接收 `int x`，在内部把 x 改成10，为什么调用者的 a 不变？

<details><summary>提示：先自己尝试</summary>

区分对象和对象值的副本。

</details>

<details><summary>参考思路与自查</summary>

形参是副本。即使传指针，也仍复制指针值，只是能通过该地址访问同一个对象。

</details>

### 2. 改错
函数 `int length(int a[]) { return sizeof(a) / sizeof(a[0]); }` 为什么不能求数组长度？

<details><summary>提示：先自己尝试</summary>

数组参数在函数中是什么类型？

</details>

<details><summary>参考思路与自查</summary>

它是指针参数；由调用者计算实际数组长度并显式传入。

</details>

### 3. 编程
写 `int is_passing(int score)`，0..59 返回0，60..100 返回1，非法成绩返回-1。

<details><summary>提示：先自己尝试</summary>

先检查有效范围，再进行等级判断。

</details>

<details><summary>参考思路与自查</summary>

至少测试 -1、0、59、60、100、101，预期 -1、0、0、1、1、-1。

</details>

### 4. 综合扩展
为 GradeSummary 增加最高分字段，保持输入数组不变，并处理空数据。

<details><summary>提示：先自己尝试</summary>

只有确认至少一个有效元素后才能使用第0项初始化最高分。

</details>

<details><summary>参考思路与自查</summary>

样例最高分90；单人成绩60的最高分为60；空数据仍 valid=0，不伪造最高分。

</details>

## 本地实践：完整 C17 程序

将下面代码保存为 `lesson.c`，执行 `cc -std=c17 -Wall -Wextra -Wpedantic lesson.c -o lesson`，再运行 `./lesson`（Windows 使用 `lesson.exe`）。此代码展示完整程序结构，不作为 Notebook 单元执行。

```c
#include <stdio.h>
struct GradeSummary {
    int valid;
    double average;
    size_t passed;
};

// 不修改输入；count 必须对应调用者提供的实际元素数。
struct GradeSummary summarize(const int scores[], size_t count) {
    struct GradeSummary result = {0, 0.0, 0};
    if (scores == NULL || count == 0 || count > 100) { return result; }
    int total = 0;
    size_t passed = 0;
    for (size_t i = 0; i < count; ++i) {
        if (scores[i] < 0 || scores[i] > 100) { return result; }
        total += scores[i];
        if (scores[i] >= 60) { ++passed; }
    }
    result.valid = 1;
    result.average = (double)total / count;
    result.passed = passed;
    return result;
}

int main(void) {
    const int scores[] = {85, 90, 58};
    size_t count = sizeof scores / sizeof scores[0];
    struct GradeSummary result = summarize(scores, count);
    if (!result.valid) {
        printf("无法统计：人数或成绩无效\n");
        return 1;
    }
    printf("平均分: %.2f, 及格: %zu/%zu\n", result.average, result.passed, count);
    return 0;
}
```

默认样例的标准输出：

```text
平均分: 77.67, 及格: 2/3
```

## 小结

清晰的函数接口说明输入、返回值和无效情况。数组参数需配合长度；递归需终止条件与有效范围。优先让计算函数只读取输入并返回结果，让调用者负责输入输出。